In [2]:
import pandas as pd
import numpy as np
import os
import re
import nltk
import spacy
from nltk.corpus import stopwords
from google.colab import drive

# Mount drive
drive.mount('/content/drive')

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')

# Load spacy model
# Run this in a separate cell first if not installed:
# !python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

# ── Paths ──────────────────────────────────────────
BASE_PATH     = "/content/drive/MyDrive/thesis_data/Original Reddit Data"
LABELLED_PATH = os.path.join(BASE_PATH, "Labelled Data")
RAW_PATH      = os.path.join(BASE_PATH, "raw data")
SAVE_PATH     = "/content/drive/MyDrive/thesis_data/"

print("Setup complete")

Mounted at /content/drive


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Setup complete


In [3]:
import glob

# ── Load Part A ────────────────────────────────────
all_files = glob.glob(os.path.join(RAW_PATH, "**", "*.csv"), recursive=True)
dfs_a = [pd.read_csv(f, low_memory=False) for f in all_files]
df_a = pd.concat(dfs_a, ignore_index=True)
print(f"Part A loaded: {df_a.shape}")

# ── Load Part B ────────────────────────────────────
label_files = {
    "Drug and Alcohol" : os.path.join(LABELLED_PATH, "LD DA 1.csv"),
    "Early Life"       : os.path.join(LABELLED_PATH, "LD EL1.csv"),
    "Personality"      : os.path.join(LABELLED_PATH, "LD PF1.csv"),
    "Trauma and Stress": os.path.join(LABELLED_PATH, "LD TS 1.csv")
}
dfs_b = []
for label, filepath in label_files.items():
    temp = pd.read_csv(filepath)
    temp['Label'] = label
    dfs_b.append(temp)
df_b = pd.concat(dfs_b, ignore_index=True)
print(f"Part B loaded: {df_b.shape}")

Part A loaded: (1851580, 8)
Part B loaded: (823, 6)


In [4]:
print("=== CLEANING PART A ===")
print(f"Starting shape: {df_a.shape}")

# ── Step 1: Filter valid subreddits ───────────────
valid_subreddits = [
    'depression', 'SuicideWatch',
    'mentalhealth', 'Anxiety', 'lonely'
]
df_a = df_a[df_a['subreddit'].isin(valid_subreddits)]
print(f"After subreddit filter:     {df_a.shape[0]:,} rows")

# ── Step 2: Remove deleted and removed posts ───────
df_a = df_a[~df_a['selftext'].isin(['[deleted]', '[removed]'])]
print(f"After removing deleted:     {df_a.shape[0]:,} rows")

# ── Step 3: Drop missing selftext ─────────────────
df_a = df_a.dropna(subset=['selftext'])
print(f"After dropping missing:     {df_a.shape[0]:,} rows")

# ── Step 4: Remove very short posts ───────────────
df_a = df_a[df_a['selftext'].str.len() >= 10]
print(f"After removing short posts: {df_a.shape[0]:,} rows")

# ── Step 5: Convert timestamps ────────────────────
df_a['date'] = pd.to_datetime(df_a['created_utc'], unit='s')
df_a['year']  = df_a['date'].dt.year
df_a['month'] = df_a['date'].dt.month
print(f"Timestamps converted")

# ── Step 6: Combine title and selftext ────────────
df_a['full_text'] = (
    df_a['title'].fillna('') + ' ' + df_a['selftext'].fillna('')
)
print(f"Title and selftext combined")

print(f"\nFinal Part A shape: {df_a.shape}")

=== CLEANING PART A ===
Starting shape: (1851580, 8)
After subreddit filter:     1,847,870 rows
After removing short posts: 1,603,198 rows
Timestamps converted
Title and selftext combined

Final Part A shape: (1603198, 12)


In [5]:
# ── Text cleaning function ─────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Lowercase
    text = text.lower()
    return text

# ── Apply to Part A ────────────────────────────────
print("Cleaning text... this may take a few minutes")
df_a['clean_text'] = df_a['full_text'].apply(clean_text)
print(f"Text cleaning done")
print(f"Sample clean text:\n{df_a['clean_text'].iloc[0][:200]}")

Cleaning text... this may take a few minutes
Text cleaning done
Sample clean text:
sad i have everything i want in life and somehow i still feel like i want to kill myself i love my girlfriend and i dont want to leave her but the thoughts that i need to die just wont stop i have no 


In [6]:
# ── Version 1: VADER text ──────────────────────────
# VADER needs minimal cleaning
# Keep punctuation — VADER uses ! and ? for intensity
# Keep stopwords — VADER needs full sentences
# Just remove URLs and HTML

def clean_for_vader(text):
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Version 2: BERT text ───────────────────────────
# BERT needs full cleaning
# Remove stopwords and lemmatise

stop_words = set(stopwords.words('english'))

def clean_for_bert(text):
    if not isinstance(text, str):
        return ''
    # Basic cleaning first
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower().strip()
    # Remove stopwords
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

# ── Apply both to Part A ───────────────────────────
print("Creating VADER text version...")
df_a['vader_text'] = df_a['full_text'].apply(clean_for_vader)

print("Creating BERT text version...")
df_a['bert_text'] = df_a['full_text'].apply(clean_for_bert)

print("Done")
print(f"\nOriginal:   {df_a['full_text'].iloc[0][:100]}")
print(f"VADER text: {df_a['vader_text'].iloc[0][:100]}")
print(f"BERT text:  {df_a['bert_text'].iloc[0][:100]}")

Creating VADER text version...
Creating BERT text version...
Done

Original:   sad i have everything i want in life and somehow i still feel like i want to kill myself. i love my 
VADER text: sad i have everything i want in life and somehow i still feel like i want to kill myself. i love my 
BERT text:  sad everything want life somehow still feel like want kill love girlfriend dont want leave thoughts 


In [7]:
print("=== CLEANING PART B ===")
print(f"Starting shape: {df_b.shape}")

# ── Drop missing rows ──────────────────────────────
df_b = df_b.dropna(subset=['selftext'])
print(f"After dropping missing: {df_b.shape[0]} rows")

# ── Drop CAT 1 column — not needed ────────────────
if 'CAT 1' in df_b.columns:
    df_b = df_b.drop(columns=['CAT 1'])
print(f"Columns kept: {df_b.columns.tolist()}")

# ── Combine title and selftext ─────────────────────
df_b['full_text'] = (
    df_b['title'].fillna('') + ' ' + df_b['selftext'].fillna('')
)

# ── Apply both cleaning versions ───────────────────
df_b['vader_text'] = df_b['full_text'].apply(clean_for_vader)
df_b['bert_text']  = df_b['full_text'].apply(clean_for_bert)

print(f"\nFinal Part B shape: {df_b.shape}")
print(f"\nLabel distribution after cleaning:")
print(df_b['Label'].value_counts())

=== CLEANING PART B ===
Starting shape: (823, 6)
After dropping missing: 800 rows
Columns kept: ['score', 'selftext', 'subreddit', 'title', 'Label']

Final Part B shape: (800, 8)

Label distribution after cleaning:
Label
Drug and Alcohol     200
Early Life           200
Personality          200
Trauma and Stress    200
Name: count, dtype: int64


In [8]:
print("=== FINAL VERIFICATION ===")

print("\nPart A:")
print(f"  Shape:            {df_a.shape}")
print(f"  Columns:          {df_a.columns.tolist()}")
print(f"  Missing vader_text: {df_a['vader_text'].isna().sum()}")
print(f"  Missing bert_text:  {df_a['bert_text'].isna().sum()}")
print(f"  Date range:       {df_a['date'].min()} to {df_a['date'].max()}")
print(f"  Subreddits:       {df_a['subreddit'].value_counts().to_dict()}")

print("\nPart B:")
print(f"  Shape:            {df_b.shape}")
print(f"  Columns:          {df_b.columns.tolist()}")
print(f"  Labels:           {df_b['Label'].value_counts().to_dict()}")

=== FINAL VERIFICATION ===

Part A:
  Shape:            (1603198, 15)
  Columns:          ['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit', 'title', 'timestamp', 'date', 'year', 'month', 'full_text', 'clean_text', 'vader_text', 'bert_text']
  Missing vader_text: 0
  Missing bert_text:  0
  Date range:       2018-12-31 13:01:28 to 2022-08-31 13:56:55
  Subreddits:       {'depression': 526605, 'SuicideWatch': 402318, 'mentalhealth': 277678, 'Anxiety': 257712, 'lonely': 138885}

Part B:
  Shape:            (800, 8)
  Columns:          ['score', 'selftext', 'subreddit', 'title', 'Label', 'full_text', 'vader_text', 'bert_text']
  Labels:           {'Drug and Alcohol': 200, 'Early Life': 200, 'Personality': 200, 'Trauma and Stress': 200}


In [9]:
print("Saving clean datasets...")

# ── Save to Google Drive ───────────────────────────
df_a.to_csv(
    os.path.join(SAVE_PATH, "clean_part_a.csv"),
    index=False
)
print(f"✓ clean_part_a.csv saved — {df_a.shape[0]:,} rows")

df_b.to_csv(
    os.path.join(SAVE_PATH, "clean_part_b.csv"),
    index=False
)
print(f"✓ clean_part_b.csv saved — {df_b.shape[0]} rows")

print("\nAll clean data saved to Google Drive")
print("You never need to run cleaning again — just load these files")

Saving clean datasets...
✓ clean_part_a.csv saved — 1,603,198 rows
✓ clean_part_b.csv saved — 800 rows

All clean data saved to Google Drive
You never need to run cleaning again — just load these files
